# CS 3892 / 5892 — Session 13 · Regular languages and Büchi automata

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-10-08-regular-languages-and-buchi-automata.ipynb)

**Thursday, October 8, 2026.** First **finite words**: regular expressions, DFAs and NFAs, and what it
means for an automaton to **accept**. Then the **same automata** read on **infinite words**, with
**Büchi acceptance**, which is how a model checker checks LTL.

The alphabet is one bit per step of session 11's agent: **d** (it deleted something) or **n** (it
did not). Every Python file asserts its own results. NuSMV (free, LGPL) is fetched by the setup cell
for section 5.

## Setup

Run these two once.

In [ ]:
# --- Setup: find the repo (clone on Colab) and define helpers ---------------
import os, re, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-10-08-regular-languages-and-buchi-automata"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

SMV = ROOT / "sessions" / SESSION / "smv"
PYD = ROOT / "sessions" / SESSION / "python"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    path = pathlib.Path(path)
    print(f"--- {path.name} " + "-" * max(0, 60 - len(path.name)))
    print(path.read_text().rstrip())
    print()

def run(path):
    """Run one Python example and stream its output. Raises if it fails."""
    path = pathlib.Path(path)
    show(path)
    r = subprocess.run([sys.executable, str(path)], capture_output=True, text=True)
    print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.rstrip(), file=sys.stderr)
        raise RuntimeError(f"{path} failed")
    return r.stdout

print("ready — helpers: show(path), run(path)")

In [ ]:
# --- NuSMV: fetch the official FBK build if it is not already installed ------
# NuSMV is free (LGPL) from nusmv.fbk.eu. (Same cell as sessions 9-12.)
# 2.7.1 is tried first; it needs glibc >= 2.38 and libedit, which an older
# Colab image may lack, so the static 2.6.0 build is the fallback. Same verdicts.
import shutil

BUILDS = [("2.7.1", "https://nusmv.fbk.eu/distrib/2.7.1/NuSMV-2.7.1-linux64.tar.xz", "J"),
          ("2.6.0", "https://nusmv.fbk.eu/distrib/NuSMV-2.6.0-linux64.tar.gz",       "z")]

def _works(exe):
    """True only if this binary actually model-checks a file -- a missing
    library makes it exit 127 with an error that still contains "NuSMV"."""
    try:
        r = subprocess.run([exe, str(SMV / "01_ends_with_d.smv")],
                           capture_output=True, text=True, timeout=60)
        return "-- specification" in r.stdout
    except Exception:
        return False

NUSMV = shutil.which("NuSMV")
if not (NUSMV and _works(NUSMV)):
    NUSMV = None
    if os.geteuid() == 0 and shutil.which("apt-get"):      # Colab runs as root
        subprocess.run("apt-get install -y -qq libedit2 >/dev/null 2>&1", shell=True)
    for ver, url, z in BUILDS:
        dest = pathlib.Path.home() / ".local" / f"nusmv-{ver}"
        dest.mkdir(parents=True, exist_ok=True)
        subprocess.run(f"curl -sSL {url} | tar -x{z} -C {dest} --strip-components=1",
                       shell=True, check=True)
        if _works(str(dest / "bin" / "NuSMV")):
            NUSMV = str(dest / "bin" / "NuSMV")
            break
        print(f"NuSMV {ver} will not start on this machine -- trying the next build")
assert NUSMV, "could not install NuSMV"
print("NuSMV:", NUSMV)

def smv(path):
    """Run NuSMV on a file in batch mode; return its output minus the banner."""
    r = subprocess.run([NUSMV, str(path)], capture_output=True, text=True)
    return "\n".join(l for l in (r.stdout + r.stderr).splitlines() if not l.startswith("***"))

def results(out):
    """Split NuSMV output into {printed formula: (verdict, counterexample text)}.
    NuSMV does not print verdicts in file order, so match them by formula."""
    res, key = {}, None
    for line in out.splitlines():
        m = re.match(r"-- (?:specification|invariant)\s+(.*?)\s+is (true|false)$", line)
        if m:
            key = re.sub(r"\s+", " ", m.group(1)).strip()
            res[key] = [m.group(2), ""]
        elif key and res[key][0] == "false":
            res[key][1] += line + "\n"
    return {k: tuple(v) for k, v in res.items()}

def verdicts(path):
    """Print one line per property, and return {formula: verdict}."""
    out = smv(path)
    res = results(out)
    for k, (v, _) in res.items():
        print(f"{v:6} {k}")
    return {k: v for k, (v, _) in res.items()}

print("ready -- smv(path), results(out), verdicts(path)")

## 1. Words, regular expressions, and a DFA

A finite log of the agent is a **word** over `{d, n}`, and a property of logs is a **language**,
which is a set of words. **END_D**, "the log ends with a delete", is a DFA with two states. It
**accepts** a word when its run **ends** in the accepting state `q1`.

In [ ]:
# END_D, and EVEN_D (an even number of deletes).
from IPython.display import SVG, display
FIG = ROOT / "sessions" / SESSION / "figures"
display(SVG(filename=str(FIG / "dfa_ends_d.svg")))
display(SVG(filename=str(FIG / "dfa_even_d.svg")))

In [ ]:
out = run(PYD / "01_regex_and_dfa.py")
assert "agree on all of them" in out and "differ on '' -- the empty word" in out

Python's `re` *is* a regular-expression engine — mostly. `fullmatch` asks whether the **whole** word
is in the language. (Backreferences such as `(a*)b\1` go beyond regular languages, so no finite
automaton can do what they do.)

In [ ]:
import re
for w in ["ndnd", "ndn", "", "ddn"]:
    print(f"{w!r:7}  ends with d: {bool(re.fullmatch(r'[dn]*d', w))!s:5}   "
          f"(n*d)*: {bool(re.fullmatch(r'(n*d)*', w))}")

## 2. Product, complement, emptiness

**Intersection** runs two DFAs in lockstep, and the product accepts when both accept.
**Complement** swaps the accepting states of a complete DFA. **Emptiness** asks whether an
accepting state is reachable, which is a graph search. Section 1's run already checked all three.
Here is the product it built:

In [ ]:
display(SVG(filename=str(FIG / "product_end_even.svg")))

## 3. Nondeterminism, and the subset construction

An **NFA** may have several moves on a letter. It accepts if **some** run ends in an accepting
state, so it can *guess*. Here it guesses which `d` is the second-to-last letter. The subset
construction (Rabin and Scott, 1959) gives an equivalent DFA whose states are **sets** of NFA states.
It can need `2^n` of them, and minimization shows that here every one of them is needed.

In [ ]:
display(SVG(filename=str(FIG / "nfa_second_last.svg")))
display(SVG(filename=str(FIG / "dfa_second_last.svg")))

In [ ]:
out = run(PYD / "02_nfa_subset_construction.py")
assert "8            9                   256           256" in out

## 4. The same automata, on infinite words

A reactive system never stops, so its runs are **infinite** words. A **Büchi** automaton is read
on one and **accepts** if some run of it visits an accepting state **infinitely often**.
END_D, unchanged, then accepts exactly the words with infinitely many `d`, which is `G F d`.

`F G n` ("eventually, no more deletes") needs a **guess**. No deterministic Büchi automaton
accepts it (Landweber, 1969), and the file checks every one with up to 3 states.
"d at every even position" is ω-regular, but **no LTL formula** expresses it (Wolper, 1983).

In [ ]:
display(SVG(filename=str(FIG / "nba_fg_n.svg")))
display(SVG(filename=str(FIG / "dba_even_pos.svg")))

In [ ]:
out = run(PYD / "03_buchi_acceptance.py")
assert "k = 3:  5832 automata   agree with F G n: 0" in out

## 5. NuSMV: an automaton is a transition system

`01_ends_with_d.smv` is END_D reading an **unconstrained** input, so its runs are *all* the infinite
words. NuSMV confirms that Büchi acceptance (`q1` infinitely often) is `G F d`. Its counterexample to
`G F (q = q1)` is a lasso word, `d (n)^ω`, whose loop contains no `d`.

In [ ]:
F = SMV / "01_ends_with_d.smv"
show(F)
res = results(smv(F))
for k, (v, _) in res.items():
    print(f"{v:6} {k}")
assert [v for v, _ in res.values()] == ["true", "true", "false"], res
cex = list(res.values())[2][1]
print(cex)
loop = cex.split("Loop starts here", 1)[1]
assert "letter = d" in cex.split("Loop starts here")[0] and "letter = d" not in loop
print("the lasso: d, then n forever -- no d in the loop")

## 6. The recipe, on the agent

To check `M ⊨ φ`: negate φ, build a Büchi automaton for `¬φ`, take the **product** with `M`, and
search it for a reachable **accepting cycle**. A cycle found is a lasso, which is a counterexample.
If there is none, the property holds. Session 12's file runs all of this by hand on the agent, and
the cell after it checks that NuSMV agrees.

In [ ]:
display(SVG(filename=str(FIG / "agent.svg")))
S12 = ROOT / "sessions" / "cs3892-2026-10-06-ctl-and-buchi-automata"
run(S12 / "python" / "04_buchi_emptiness.py")
agent = results(smv(S12 / "smv" / "01_ctl_on_the_agent.smv"))
assert agent["G ( F del)"][0] == "false" and agent["G (del -> approved)"][0] == "true"
print("NuSMV agrees: G F del fails, G (del -> approved) holds")

**In smvis** ([bit.ly/fmaiv_smvis](https://bit.ly/fmaiv_smvis)), paste session 12's
`01_ctl_on_the_agent.smv` and use **LTL Buchi Analysis** → `G (F (del))` → **Compose & Analyze**.
smvis builds the automaton for `F (G (!del))`, which is section 4's `F G n` automaton, and draws the
accepting cycle on the state graph.

## Try it yourself

- Write a regular expression for "never two deletes in a row", and a DFA for it. Then use
  `01`'s symmetric-difference check to test that they agree.
- In `02`, how many states does the subset construction give for "the 3rd letter from the end is `d`"?
  Predict it, then run it.
- In `03`, build a Büchi automaton for `G (d -> F n)` ("every delete is eventually followed by a
  quiet step") and test it on the lassos. Is a deterministic one possible?